# Schrödinger equation: bound states

Stationary Schrödinger equation, in units $\hbar = m = 1$:

$$
-\tfrac{1}{2} \Delta \psi + V \psi = E \, \psi
$$

Weak form: find $\psi \in H^1_0(\Omega)$ and $E$ such that

$$
\int_\Omega \tfrac{1}{2} \nabla \psi \cdot \nabla \varphi + V \psi \varphi \, dx = E \int_\Omega \psi \, \varphi \, dx \qquad \forall \, \varphi
$$

Galerkin gives a generalized matrix eigenvalue problem $A x = E \, M x$, with the stiffness matrix $A$ and the mass matrix $M$.

In [ ]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw
import numpy as np
import scipy.sparse.linalg as sla
import matplotlib.pyplot as plt

## Check: the harmonic oscillator

$V = \frac{1}{2} |x|^2$ in 2D has the eigenvalues $E = n_x + n_y + 1$, i.e. $1, 2, 2, 3, 3, 3, \ldots$. We cut the domain at radius 7, where the eigenfunctions are tiny:

In [ ]:
mesh = Mesh(OCCGeometry(Circle((0,0), 7).Face(), dim=2).GenerateMesh(maxh=0.5)).Curve(4)

fes = H1(mesh, order=5, dirichlet=".*")
psi, phi = fes.TnT()
V = 0.5*(x*x+y*y)
a = BilinearForm(0.5*grad(psi)*grad(phi)*dx + V*psi*phi*dx).Assemble()
m = BilinearForm(psi*phi*dx).Assemble()

# scipy's shift-and-invert Lanczos works with the NGSolve matrices,
# the inverse on the free dofs is NGSolve's sparse factorization
inv = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky")
E, vecs = sla.eigsh(a.mat, k=10, M=m.mat, sigma=0, OPinv=inv)
print (E)

gfpsi = GridFunction(fes, multidim=10)
for i in range(10):
    gfpsi.vecs[i].FV().NumPy()[:] = vecs[:,i]

In [ ]:
Draw (gfpsi, mesh, animate=True, min=-0.5, max=0.5);

## Quantum billiards

A particle in a box, $V = 0$ and $\psi = 0$ on the boundary. In the circle the classical motion is integrable, in the Bunimovich stadium it is chaotic.
We use a quarter of each, which picks one symmetry class of eigenfunctions. The circle has the radius 1.5, then both have about the same area, and the same density of levels:

In [ ]:
def QuarterStadium(a=1):
    rect = MoveTo(0,0).Rectangle(a,1).Face()
    quarter = Circle((a,0),1).Face() * MoveTo(a,0).Rectangle(1,1).Face()
    return rect + quarter

def QuarterCircle(r=1.5):
    return Circle((0,0),r).Face() * MoveTo(0,0).Rectangle(r,r).Face()

Draw (QuarterStadium());

We need many eigenvalues. As before, scipy's shift-and-invert Lanczos with NGSolve matrices and NGSolve's factorization computes the lowest 300 eigenpairs of $-\Delta$ in the stadium:

In [ ]:
def Assemble(shape, maxh=0.05, order=4):
    mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=maxh)).Curve(order)
    fes = H1(mesh, order=order, dirichlet=".*")
    u,v = fes.TnT()
    a = BilinearForm(grad(u)*grad(v)*dx).Assemble()
    m = BilinearForm(u*v*dx).Assemble()
    return fes, a.mat, m.mat

def Billiard(shape, num=300):
    fes, amat, mmat = Assemble(shape)
    inv = amat.Inverse(fes.FreeDofs(), inverse="sparsecholesky")
    lam, vecs = sla.eigsh(amat, k=num, M=mmat, sigma=0, OPinv=inv)
    modes = GridFunction(fes, multidim=num)
    for i in range(num):
        modes.vecs[i].FV().NumPy()[:] = vecs[:,i]
    return fes.mesh, lam, modes

with TaskManager():
    mesh_s, lam_s, modes_s = Billiard(QuarterStadium())
print ("stadium:", lam_s[:5], "...", lam_s[-1])

Some eigenfunctions of the stadium, low and high ones. Use the `multidim` slider to walk through them:

In [ ]:
def Select(modes, indices):
    gf = GridFunction(modes.space, multidim=len(indices))
    for k, i in enumerate(indices):
        gf.vecs[k].data = modes.vecs[int(i)]
    return gf

show = Select(modes_s, [0,1,2,3,50,100,150,200,250,299])
Draw (show, mesh_s, animate=True, order=3, min=-2, max=2,
      settings={"Colormap" : { "ncolors" : 64 }});

## Bouncing-ball modes

A typical eigenfunction of a chaotic billiard spreads over the whole domain. Then the rectangular part carries the fraction $|\Omega_{rect}| / |\Omega| \approx 0.56$ of $\int \psi^2$.
Some modes do not follow this rule: the **bouncing-ball modes** stay between the straight walls, where the classical orbits bounce up and down:

In [ ]:
inrect = IfPos(1-x, 1, 0)
gf = GridFunction(modes_s.space)
weight = []
for vec in modes_s.vecs:
    gf.vec.data = vec
    weight.append (Integrate(inrect*gf*gf, mesh_s, order=8))
weight = np.array(weight)

plt.figure(figsize=(8,3))
plt.plot(weight, "o", ms=3)
plt.axhline(1/(1+pi/4), color="k", ls="--")
plt.xlabel("mode number"); plt.ylabel("weight in the rectangle")
plt.show()

The six modes with the largest weight in the rectangle:

In [ ]:
bb = np.argsort(-weight)[:6]
print ("modes", bb, "weights", weight[bb].round(2))
Draw (Select(modes_s, bb), mesh_s, animate=True, order=3, min=-2, max=2,
      settings={"Colormap" : { "ncolors" : 64 }});

## Level spacing statistics

Statistics need more levels, we take 1000 for each billiard. Shift-and-invert Lanczos finds the eigenvalues next to a shift $\sigma$. By Weyl's law the levels have the constant density $|\Omega| / 4 \pi$. So we cut the spectrum into slices of 100 levels, and use one shift per slice.
Each slice needs one sparse factorization of $A - \sigma M$:

In [ ]:
def Levels(shape, num=1000, perslice=100):
    fes, amat, mmat = Assemble(shape, order=6)
    shifted = amat.CreateMatrix()
    width = perslice * 4*pi / Integrate(1, fes.mesh)
    lam, left = [], 0
    while len(lam) < num:
        sigma = left + width/2
        shifted.AsVector().data = amat.AsVector() - sigma*mmat.AsVector()
        inv = shifted.Inverse(fes.FreeDofs(), inverse="sparsecholesky")
        mu = sla.eigsh(amat, k=2*perslice, M=mmat, sigma=sigma, OPinv=inv, return_eigenvectors=False)
        lam += [l for l in mu if left <= l < left+width]
        left += width
    return fes.mesh, np.sort(lam)[:num]

with TaskManager():
    levels = { "quarter circle" : Levels(QuarterCircle()),
               "quarter stadium" : Levels(QuarterStadium()) }

Unfold the spectrum with Weyl's law, $N(\lambda) \approx \frac{|\Omega|}{4 \pi} \lambda - \frac{|\partial \Omega|}{4 \pi} \sqrt{\lambda}$, so that the mean spacing is 1.
Integrable systems show Poisson statistics $P(s) = e^{-s}$, chaotic ones level repulsion, close to the Wigner surmise $P(s) = \frac{\pi s}{2} e^{-\pi s^2/4}$ (Bohigas, Giannoni, Schmit 1984):

In [ ]:
def Spacings(mesh, lam, skip=30):
    area = Integrate(1, mesh)
    length = Integrate(1, mesh, BND)
    N = area/(4*pi)*lam - length/(4*pi)*np.sqrt(lam)
    return np.diff(N[skip:])

s = np.linspace(0,4,200)
fig, ax = plt.subplots(1, 2, figsize=(10,3.5), sharey=True)
for axi, (name, (mesh, lam)) in zip(ax, levels.items()):
    axi.hist(Spacings(mesh, lam), bins=20, range=(0,4), density=True, alpha=0.6)
    axi.plot(s, np.exp(-s), label="Poisson")
    axi.plot(s, np.pi*s/2*np.exp(-np.pi*s**2/4), label="Wigner")
    axi.set_title(name); axi.set_xlabel("s"); axi.legend()
plt.show()

## Accuracy

The eigenvalues converge with the rate $h^{2p}$. High order elements on curved meshes give many accurate eigenvalues from few unknowns.
For the circle we know the exact values: $\lambda = (j_{m,n} / r)^2$ with the zeros $j_{m,n}$ of the Bessel functions $J_m$, where $m = 2, 4, 6, \ldots$ for the quarter circle:

In [ ]:
from scipy.special import jn_zeros

mesh, lam = levels["quarter circle"]
zeros = np.sort(np.concatenate([jn_zeros(m, 50) for m in range(2, 140, 2)]))[:len(lam)]
exact = (zeros/1.5)**2
print ("largest relative error of the", len(lam), "eigenvalues:", np.max(np.abs(lam-exact)/exact))
print ("in units of the mean level spacing:  ", np.max(np.abs(lam-exact)) / (4*pi/Integrate(1, mesh)))

## Extensions

* a potential $V$, or a magnetic field: $\frac{1}{2} (-i \nabla - A) \psi \cdot \overline{(-i \nabla - A) \varphi}$ with `complex=True`
* 3D, and many eigenvalues with block-iterative solvers (`solvers.LOBPCG`, `solvers.PINVIT`, FEAST-type contour integrals)
* periodic boundary conditions (`Periodic(H1(...), phase=...)`) for Bloch waves